# PRODUCTION

In [1]:
### ==== IMPORT ====
import pypdf
import logging
from pathlib import Path
import pandas as pd
import re
import spacy

# Load the NLP model - with installation fallback
try:
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy model loaded successfully!")
except OSError:
    print("⚠ spaCy model not found.")
    print("  To fix this, run in terminal: python -m spacy download en_core_web_sm")
    print("  For now, extraction will use pattern matching (reduced accuracy).")
    nlp = None

# Suppress all pypdf log messages below the ERROR level
logging.getLogger("pypdf").setLevel(logging.ERROR)

✓ spaCy model loaded successfully!


In [2]:
### ==== HELPTER FUNCTIONS ====
def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "pyproject.toml").exists() or (candidate / "README.md").exists():
            return candidate
    return start

def extract_authors(author_str):
    # Handle empty or NaN values
    if pd.isna(author_str) or author_str == "":
        return []
    
    # Split multiple authors by the pipe
    authors = [a.strip() for a in str(author_str).split('|')]
    
    clean_names = []
    for author in authors:
        if ',' in author:
            # Split "sir name, name" -> ["sir name", "name"]
            parts = author.split(',')
            # Reorder to "name sir name" (taking second part first)
            clean_names.append(f"{parts[1].strip()} {parts[0].strip()}")
        else:
            # If no comma exists, just keep the name as is
            clean_names.append(author.strip())
            
    return clean_names


In [3]:
### ==== FILE PATHS ====
# Resolve the repository root (folder containing .git), then point to your PDF directory.
project_root = locate_project_root(Path.cwd())
print(f"Project root: {project_root}")

# Update this relative path if your PDFs are stored elsewhere in the repo
folder_path = project_root / "Data/RAW_test/"

pdf_files = sorted(folder_path.glob("*.pdf"))

if not pdf_files:
    print(f"No PDF files found in: {folder_path}")
else:
    total_files = len(pdf_files)
    print(f"Found {total_files} PDF file(s) in: {folder_path}")

    user_input = input(f"How many files do you want to process? (1-{total_files}): ").strip()

    try:
        num_to_process = int(user_input)
        if num_to_process <= 0:
            raise ValueError
    except ValueError:
        raise SystemExit("Error: Invalid input. Please enter a positive integer. Execution terminated.")

    if num_to_process > total_files:
        choice = input(
            f"Error: Requested {num_to_process} files, but only {total_files} available.\n"
            "Type 'all' to process all files, or 'q' to terminate: "
        ).strip().lower()

        if choice == "all":
            num_to_process = total_files
        elif choice in {"q", "quit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")
        else:
            raise SystemExit("Error: Invalid choice. Execution terminated without processing files.")

Project root: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR
Found 23 PDF file(s) in: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/Data/RAW_test


In [4]:
### ==== HELPER FILES ====
csv_file = "../Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis.csv"
df = pd.read_csv(csv_file, sep=";", encoding="utf-8")

# Clean the author names
df['clean_authors'] = df['Author'].apply(extract_authors)

# Create your lookup dictionary
author_lookup = dict(zip(df['pdf_file'], df['clean_authors']))


def canonicalize_for_validation(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip().lower()
    text = re.sub(r"[^a-zæøå\s\-\.'’]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def get_authors_for_file_validation(filename: str):
    file_name_only = Path(filename).name
    file_stem = Path(file_name_only).stem
    matched_authors = []

    for key, authors in author_lookup.items():
        key_name = Path(str(key)).name
        key_stem = Path(key_name).stem
        if key == file_name_only or key_name == file_name_only or key_stem == file_stem:
            matched_authors.extend(authors)

    return {canonicalize_for_validation(a) for a in matched_authors if canonicalize_for_validation(a)}


def main(pdf_path):
    return _main_impl(pdf_path)

In [5]:
### ==== FUNCTIONS ====
def extract_supervisors(reader, max_pages=10, pdf_filename=None):

    supervisor_keywords = {
        "supervisor", "supervisors", "advisor", "advisors", "adviser", "advisers",
        "vejleder", "vejledere", "thesis supervisor", "project supervisor", "guided by",
        "speciale vejleder", "speciale supervisor", "supervised by"
    }
    affiliation_keywords = {
        "university", "department", "institute", "faculty", "school", "center", "centre",
        "dtu", "technical university", "danmarks tekniske universitet",
        "technical university of denmark"
    }

    def normalize(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip(" -:\t")

    def canonicalize_name(text: str) -> str:
        text = normalize(text).lower()
        text = re.sub(r"[^a-zæøå\s\-\.'’]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_authors_for_file(filename: str | None):
        if not filename or "author_lookup" not in globals():
            return set()

        file_name_only = Path(filename).name
        file_stem = Path(file_name_only).stem
        matched_authors = []

        for key, authors in author_lookup.items():
            key_name = Path(str(key)).name
            key_stem = Path(key_name).stem
            if key == file_name_only or key_name == file_name_only or key_stem == file_stem:
                matched_authors.extend(authors)

        return {canonicalize_name(a) for a in matched_authors if canonicalize_name(a)}

    file_authors = get_authors_for_file(pdf_filename)

    def is_author_name(text: str) -> bool:
        return canonicalize_name(text) in file_authors

    def looks_like_name(text: str) -> bool:
        text = normalize(text)
        if not text or any(ch.isdigit() for ch in text):
            return False
        tokens = re.findall(r"[A-Za-zÆØÅæøå][A-Za-zÆØÅæøå\-\.'’]*", text)
        if not (2 <= len(tokens) <= 6):
            return False
        capitalized = sum(
            1 for t in tokens
            if re.match(r"^[A-ZÆØÅ][a-zæøåA-ZÆØÅ\-\.'’]*$", t) or re.match(r"^[A-ZÆØÅ]\.$", t)
        )
        return capitalized >= 2

    candidates = []
    seen = set()

    pages_to_scan = min(max_pages, len(reader.pages))
    for page_idx in range(pages_to_scan):
        text = reader.pages[page_idx].extract_text() or ""
        lines = [normalize(line) for line in text.splitlines() if normalize(line)]

        for i, line in enumerate(lines):
            lower = line.lower()

            keyword_hit = any(k in lower for k in supervisor_keywords)
            if not keyword_hit:
                continue

            # Try extracting right side after labels like "Supervisor: John Doe"
            m = re.search(
                r"(supervisors?|advis(?:or|er)s?|vejled(?:er|ere))\s*[:\-]\s*(.+)$",
                line,
                flags=re.IGNORECASE,
            )
            if m:
                value = normalize(m.group(2))
                if value and value.lower() not in seen and not is_author_name(value):
                    seen.add(value.lower())
                    candidates.append(f"p.{page_idx + 1}: {value}")

            # Include nearby context lines (name + affiliation often split across lines)
            for j in range(max(0, i - 2), min(len(lines), i + 3)):
                ctx = lines[j]
                ctx_lower = ctx.lower()

                if (
                    any(k in ctx_lower for k in supervisor_keywords)
                    or any(k in ctx_lower for k in affiliation_keywords)
                    or looks_like_name(ctx)
                ):
                    key = ctx_lower
                    if key not in seen and not is_author_name(ctx):
                        seen.add(key)
                        candidates.append(f"p.{page_idx + 1}: {ctx}")

    return candidates


def extract_supervisors_refined(reader, max_pages=10, pdf_filename=None):
    if nlp is None:
        print("Warning: Using pattern-based extraction (spaCy model not available)")
        return extract_supervisors_pattern_based(reader, max_pages=max_pages, pdf_filename=pdf_filename)

    not_names = [
        "DTU",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "I want to thank",
    ]
    not_names_normalized = {x.lower().strip() for x in not_names}

    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def canonicalize_name(text: str) -> str:
        text = normalize_spaces(text).lower()
        text = re.sub(r"[^a-zæøå\s\-\.'’]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_authors_for_file(filename: str | None):
        if not filename or "author_lookup" not in globals():
            return set()

        file_name_only = Path(filename).name
        file_stem = Path(file_name_only).stem
        matched_authors = []

        for key, authors in author_lookup.items():
            key_name = Path(str(key)).name
            key_stem = Path(key_name).stem
            if key == file_name_only or key_name == file_name_only or key_stem == file_stem:
                matched_authors.extend(authors)

        return {canonicalize_name(a) for a in matched_authors if canonicalize_name(a)}

    file_authors = get_authors_for_file(pdf_filename)

    def is_author_name(text: str) -> bool:
        return canonicalize_name(text) in file_authors

    def is_valid_name_candidate(text: str) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        if candidate.lower() in not_names_normalized:
            return False

        if is_author_name(candidate):
            return False

        return True

    titles_pattern = r"\b(Prof\.|Professor|Associate [Pp]rofessor|Assistant [Pp]rofessor|Ph\.D\.|PhD|MSc|BSc|Vejleder|Advisor|Supervisor|Adviser)\b"

    found_data = []
    seen_names = set()

    pages_to_scan = min(max_pages, len(reader.pages))
    for page_idx in range(pages_to_scan):
        text = reader.pages[page_idx].extract_text() or ""

        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                raw_name = ent.text.strip().replace("\n", " ")
                clean_name = re.sub(titles_pattern, "", raw_name, flags=re.IGNORECASE).strip(" ,.-")

                if not is_valid_name_candidate(clean_name):
                    continue

                if clean_name.lower() not in seen_names:
                    seen_names.add(clean_name.lower())
                    found_data.append({
                        "name": clean_name,
                        "page": page_idx + 1
                    })

    return found_data


def extract_supervisors_pattern_based(reader, max_pages=10, pdf_filename=None):
    """Fallback extraction using pattern matching (simplified, less accurate)."""
    supervisor_keywords = {
        "supervisor", "supervisors", "advisor", "advisors", "adviser", "advisers",
        "vejleder", "vejledere", "thesis supervisor", "project supervisor", "guided by",
        "speciale vejleder", "speciale supervisor", "supervised by"
    }

    not_names = [
        "DTU",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "I want to thank",
    ]
    not_names_normalized = {x.lower().strip() for x in not_names}

    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def canonicalize_name(text: str) -> str:
        text = normalize_spaces(text).lower()
        text = re.sub(r"[^a-zæøå\s\-\.'’]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_authors_for_file(filename: str | None):
        if not filename or "author_lookup" not in globals():
            return set()

        file_name_only = Path(filename).name
        file_stem = Path(file_name_only).stem
        matched_authors = []

        for key, authors in author_lookup.items():
            key_name = Path(str(key)).name
            key_stem = Path(key_name).stem
            if key == file_name_only or key_name == file_name_only or key_stem == file_stem:
                matched_authors.extend(authors)

        return {canonicalize_name(a) for a in matched_authors if canonicalize_name(a)}

    file_authors = get_authors_for_file(pdf_filename)

    def is_author_name(text: str) -> bool:
        return canonicalize_name(text) in file_authors

    def is_valid_name_candidate(text: str) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        if candidate.lower() in not_names_normalized:
            return False

        if is_author_name(candidate):
            return False

        return True

    found_data = []
    seen_names = set()

    pages_to_scan = min(max_pages, len(reader.pages))
    for page_idx in range(pages_to_scan):
        text = reader.pages[page_idx].extract_text() or ""
        lines = text.splitlines()

        for line in lines:
            line_lower = line.lower()
            if any(k in line_lower for k in supervisor_keywords):
                if ':' in line:
                    parts = line.split(':', 1)
                    candidate = parts[1].strip()
                elif '-' in line:
                    parts = line.split('-', 1)
                    candidate = parts[1].strip()
                else:
                    candidate = line

                if is_valid_name_candidate(candidate):
                    normalized = normalize_spaces(candidate).lower()
                    if normalized not in seen_names:
                        seen_names.add(normalized)
                        found_data.append({
                            "name": normalize_spaces(candidate),
                            "page": page_idx + 1
                        })

    return found_data


def extract_supervisors_hybrid(reader, max_pages=10, pdf_filename=None):
    """Hybrid approach: Use pattern-based extraction, then refine with NLP."""
    not_names = [
        "DTU",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "I want to thank",
        "Universitet Danmarks Tekniske Universitet",
        "The Technical University of Denmark",
    ]
    not_names_normalized = {x.lower().strip() for x in not_names}

    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def canonicalize_name(text: str) -> str:
        text = normalize_spaces(text).lower()
        text = re.sub(r"[^a-zæøå\s\-\.'’]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_authors_for_file(filename: str | None):
        if not filename or "author_lookup" not in globals():
            return set()

        file_name_only = Path(filename).name
        file_stem = Path(file_name_only).stem
        matched_authors = []

        for key, authors in author_lookup.items():
            key_name = Path(str(key)).name
            key_stem = Path(key_name).stem
            if key == file_name_only or key_name == file_name_only or key_stem == file_stem:
                matched_authors.extend(authors)

        return {canonicalize_name(a) for a in matched_authors if canonicalize_name(a)}

    file_authors = get_authors_for_file(pdf_filename)

    def is_author_name(text: str) -> bool:
        return canonicalize_name(text) in file_authors

    def is_valid_name_candidate(text: str) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        if candidate.lower() in not_names_normalized:
            return False

        if is_author_name(candidate):
            return False

        return True

    if nlp is None:
        print("Note: spaCy model not available. Using pattern-based extraction only.")
        return extract_supervisors_pattern_based(reader, max_pages=max_pages, pdf_filename=pdf_filename)

    pattern_candidates = extract_supervisors(reader, max_pages=max_pages, pdf_filename=pdf_filename)
    titles_pattern = r"\b(Prof\.|Professor|Associate [Pp]rofessor|Assistant [Pp]rofessor|Ph\.D\.|PhD|MSc|BSc|Vejleder|Advisor|Supervisor|Adviser)\b"

    refined_data = []
    seen_names = set()

    for candidate_text in pattern_candidates:
        parts = candidate_text.split(": ", 1)
        page_info = parts[0]
        text_to_process = parts[1] if len(parts) > 1 else candidate_text
        page_num = int(page_info.split(".")[-1])

        doc = nlp(text_to_process)

        person_found = False
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                raw_name = ent.text.strip().replace("\n", " ")
                clean_name = re.sub(titles_pattern, "", raw_name, flags=re.IGNORECASE).strip(" ,.-")

                if not is_valid_name_candidate(clean_name):
                    continue

                if clean_name.lower() not in seen_names:
                    seen_names.add(clean_name.lower())
                    refined_data.append({
                        "name": clean_name,
                        "page": page_num,
                        "source": "hybrid"
                    })
                    person_found = True

        normalized_raw = normalize_spaces(text_to_process)
        if not person_found and is_valid_name_candidate(normalized_raw):
            lowered = normalized_raw.lower()
            if lowered not in seen_names and lowered not in not_names_normalized:
                seen_names.add(lowered)
                refined_data.append({
                    "name": normalized_raw,
                    "page": page_num,
                    "source": "pattern_only"
                })

    return refined_data




In [6]:
### ==== MAIN FUNCTION IMPLEMENTATION ====
# Open and read in a LOCAL PDF file
def _main_impl(pdf_path): # , supres_prints=True):
    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def canonicalize_name(text: str) -> str:
        text = normalize_spaces(text).lower()
        text = re.sub(r"[^a-zæøå\s\-\.'’]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_authors_for_file(filename: str | None):
        if not filename or "author_lookup" not in globals():
            return set()

        file_name_only = Path(filename).name
        file_stem = Path(file_name_only).stem
        matched_authors = []

        for key, authors in author_lookup.items():
            key_name = Path(str(key)).name
            key_stem = Path(key_name).stem
            if key == file_name_only or key_name == file_name_only or key_stem == file_stem:
                matched_authors.extend(authors)

        return {canonicalize_name(a) for a in matched_authors if canonicalize_name(a)}

    not_names = {
        "dtu",
        "technical university of denmark",
        "danmarks tekniske universitet",
        "vil jeg gerne takke",
        "i want to thank",
        "universitet danmarks tekniske universitet",
        "the technical university of denmark",
        "analytics product owner",
        "supervised by",
        "exploring anchored text indexing",
        "nmbu 101577 / dtu s144493"
    }
    not_names = {x.lower().strip() for x in not_names}

    non_name_terms = {
        "director", "department", "faculty", "university", "universitet", "institute", "institut",
        "school", "center", "centre", "head", "group", "laboratory", "lab", "chair", "professor", "phd",
        "post doc", "researcher", "student", "assistant", "associate", "vejleder", "advisor", "supervisor",
        "ph.d.", "a/s"
    }
    non_name_terms = {x.lower().strip() for x in non_name_terms}

    related_exclusion_terms = {
        "master thesis", "education"
    }
    related_exclusion_terms = {x.lower().strip() for x in related_exclusion_terms}

    def is_valid_name(text: str, file_authors: set[str]) -> bool:
        candidate = normalize_spaces(text).strip(" ,.-:;\t")
        if not candidate:
            return False

        words = candidate.split()
        if len(words) < 2 or len(words) > 6:
            return False

        candidate_norm = candidate.lower()
        if candidate_norm in not_names:
            return False

        if ':' in candidate:
            return False

        if any(term in candidate_norm for term in related_exclusion_terms):
            return False

        # Reject role/organization phrases that are not person names.
        if any(term in candidate_norm for term in non_name_terms):
            return False

        if canonicalize_name(candidate) in file_authors:
            return False

        return True

    def clean_supervisor_candidates(raw_candidates, filename: str):
        file_authors = get_authors_for_file(filename)
        titles_pattern = r"\b(Prof\.|Professor|Associate [Pp]rofessor|Assistant [Pp]rofessor|Ph\.D\.|PhD|MSc|BSc|Vejleder|Advisor|Supervisor|Adviser)\b"

        def name_tokens(name: str):
            return re.findall(r"[A-Za-zÆØÅæøå]+", name.lower())

        def token_match(a: str, b: str) -> bool:
            return a == b or (len(a) == 1 and b.startswith(a)) or (len(b) == 1 and a.startswith(b))

        def likely_same_person(name_a: str, name_b: str) -> bool:
            ta = name_tokens(name_a)
            tb = name_tokens(name_b)
            if len(ta) < 2 or len(tb) < 2:
                return False

            # Require first and last name compatibility.
            if not token_match(ta[0], tb[0]):
                return False
            if not token_match(ta[-1], tb[-1]):
                return False

            # Only merge when middle-name structure is comparable.
            ma = ta[1:-1]
            mb = tb[1:-1]
            if len(ma) != len(mb):
                return False
            return all(token_match(x, y) for x, y in zip(ma, mb))

        def name_specificity(name: str):
            tokens = name_tokens(name)
            full_tokens = sum(1 for t in tokens if len(t) > 1)
            initials = sum(1 for t in tokens if len(t) == 1)
            return (full_tokens, -initials, len(name))

        def deduplicate_name_variants(items):
            merged = []
            for item in items:
                match_idx = -1
                for i, existing in enumerate(merged):
                    if likely_same_person(item["name"], existing["name"]):
                        match_idx = i
                        break

                if match_idx == -1:
                    merged.append(item.copy())
                    continue

                existing = merged[match_idx]
                if name_specificity(item["name"]) > name_specificity(existing["name"]):
                    existing["name"] = item["name"]

                # Keep earliest page and prefer hybrid source when available.
                if item.get("page") is not None and existing.get("page") is not None:
                    existing["page"] = min(existing["page"], item["page"])
                if existing.get("source") != "hybrid" and item.get("source") == "hybrid":
                    existing["source"] = "hybrid"

            return merged

        cleaned = []
        seen = set()

        for item in raw_candidates:
            raw_name = item.get("name", "")
            page = item.get("page")
            source = item.get("source", "hybrid")

            # Rule 1: split on comma first (name vs title/position/org part)
            comma_parts = [p.strip() for p in raw_name.split(",") if p.strip()]

            # Rule 2: split each part on 'og' or 'and' or '&'
            split_parts = []
            for part in comma_parts:
                split_parts.extend([p.strip() for p in re.split(r"\b(?:og|and)\b|&", part, flags=re.IGNORECASE) if p.strip()])

            for part in split_parts:
                candidates_from_part = []

                # Run NLP on each split part
                doc = nlp(part) if nlp is not None else None
                if doc is not None:
                    for ent in doc.ents:
                        if ent.label_ == "PERSON":
                            clean_name = re.sub(titles_pattern, "", ent.text, flags=re.IGNORECASE).strip(" ,.-")
                            if clean_name:
                                candidates_from_part.append(clean_name)

                # If NLP found no PERSON in this part, keep the part itself as fallback candidate
                if not candidates_from_part:
                    fallback = re.sub(titles_pattern, "", part, flags=re.IGNORECASE).strip(" ,.-")
                    if fallback:
                        candidates_from_part.append(fallback)

                for candidate in candidates_from_part:
                    if not is_valid_name(candidate, file_authors):
                        continue

                    key = canonicalize_name(candidate)
                    if key in seen:
                        continue

                    seen.add(key)
                    cleaned.append({
                        "name": normalize_spaces(candidate),
                        "page": page,
                        "source": source,
                    })

        return deduplicate_name_variants(cleaned)

    with open(pdf_path, 'rb') as file:
        try:
            reader = pypdf.PdfReader(file)
        except Exception as e:
            print(f"Error reading PDF file: {e}")
            return None, [], []

        # Use hybrid approach and exclude any candidate that matches a known author for this file.
        supervisors_candidates = extract_supervisors_hybrid(
            reader,
            max_pages=10,
            pdf_filename=pdf_path.name,
        )

        # Additional cleaning: split and NLP-process comma/og/and composite strings.
        supervisors_candidates = clean_supervisor_candidates(supervisors_candidates, pdf_path.name)

        # Additional names-only output.
        supervisor_names_only = [item["name"] for item in supervisors_candidates if "name" in item]

        print(f"Supervisor names: {supervisor_names_only}")

    return reader, supervisors_candidates, supervisor_names_only

In [9]:
# Building dataframe to store the metrics for each PDF file
extensive_metrics_df = pd.DataFrame(columns=["pdf_file", "member_id_ss_metrics", "supervisor(s)"])

validation_failures = []

# Process the specified number of PDF files and collect metrics.
for current_file_number, pdf_path in enumerate(pdf_files[:num_to_process], start=1):
    print(f"\n=== Processing file {current_file_number} of {num_to_process} ===")
    print(f"Current file is: {pdf_path.name}\n")

    reader, supervisors_candidates, supervisor_names_only = main(pdf_path)

    # Validation: ensure returned names are not present in author_lookup for this file.
    file_authors = get_authors_for_file_validation(pdf_path.name)
    overlapping_names = [
        n for n in supervisor_names_only
        if canonicalize_for_validation(n) in file_authors
    ]

    if overlapping_names:
        validation_failures.append({
            "pdf_file": pdf_path.name,
            "overlapping_names": overlapping_names,
        })
        print(f"\nVALIDATION FAIL: names overlap with author_lookup -> {overlapping_names}")

if validation_failures:
    print("\nValidation summary: FAIL")
    for failure in validation_failures:
        print(f"- {failure['pdf_file']}: {failure['overlapping_names']}")
else:
    print("\nValidation summary: PASS for all processed files")


=== Processing file 1 of 23 ===
Current file is: 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf

Supervisor names: ['Ole Ravn', 'Masashi Sugiyama']

=== Processing file 2 of 23 ===
Current file is: 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf

Supervisor names: ['Pawel Wargocki']

=== Processing file 3 of 23 ===
Current file is: 5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf

Supervisor names: ['Anne Simone Dederichs']

=== Processing file 4 of 23 ===
Current file is: 5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf

Supervisor names: ['Paolo Masulli', 'Tobias Andersen']

=== Processi

# TODO

- [] run manual check of 23 local files and check if the correct names and number of supervisor(s) are retruned.
- [] rewrite to a .py file for GCP connection
- [] validate GCP .py connection and production mode in --test is running well
- [] continue with next metric to implement locally then to GCP


# ARCHIVES

In [ ]:
file_to_check = df["pdf_file"].iloc[29]  # Example: checking the first file in the dataframe
search_author = df["clean_authors"].iloc[29][1]  # Example: checking the first author of the first file

if file_to_check in author_lookup:
    if search_author in author_lookup[file_to_check]:
        print(f"Found {search_author} in {file_to_check}")
    else:
        print(f"Author not found for this file.")
else:
    print("File not found in the list.")

In [ ]:
#display(df[df["clean_authors"].apply(lambda x: len(x) > 1)])

In [ ]:
#print(f"validation_failures_count: {len(validation_failures)}")
#if validation_failures:
#    print(validation_failures)
#else:
#    print("All returned supervisor names passed author_lookup validation.")